In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values per column:")
print(df.isnull().sum())

Dataset Shape: (1200, 14)

First 5 rows:
     OrderID        Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000  2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001  2024-08-23     C75739    Phone         2     151.35   
2  ORD200002  2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003  2023-10-15     C33540    Chair         1     273.19   
4  ORD200004  2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0     928 Main St    Debit Card     Shipped    TRK37947903            7   
1     823 Main St        Online     Shipped    TRK91186779            3   
2     512 Main St   Credit Card   Cancelled    TRK42903982            8   
3     275 Main St    Debit Card    Returned    TRK62788070            5   
4     668 Main St        Online   Delivered    TRK29241424            8   

  CouponCode ReferralSource  TotalPrice  
0     SAVE10      Instagram     2853.10  
1     SAV

In [4]:
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

print("Remaining missing values in CouponCode:", df['CouponCode'].isnull().sum())
print("\nUnique coupon values:")
print(df['CouponCode'].value_counts())

Remaining missing values in CouponCode: 0

Unique coupon values:
CouponCode
FREESHIP     313
NO_COUPON    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64


In [5]:
for col in ['UnitPrice', 'TotalPrice']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("Outliers capped for UnitPrice and TotalPrice!")
print(df[['UnitPrice', 'TotalPrice']].describe())

Outliers capped for UnitPrice and TotalPrice!
         UnitPrice   TotalPrice
count  1200.000000  1200.000000
mean    356.412750  1053.643183
std     197.177146   818.937858
min      11.390000    11.390000
25%     186.062500   410.520000
50%     364.210000   823.615000
75%     521.570000  1578.475000
max     699.930000  3330.407500


In [6]:
for col in ['UnitPrice', 'TotalPrice']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("Outliers capped for UnitPrice and TotalPrice!")
print(df[['UnitPrice', 'TotalPrice']].describe())

Outliers capped for UnitPrice and TotalPrice!
         UnitPrice   TotalPrice
count  1200.000000  1200.000000
mean    356.412750  1053.643183
std     197.177146   818.937858
min      11.390000    11.390000
25%     186.062500   410.520000
50%     364.210000   823.615000
75%     521.570000  1578.475000
max     699.930000  3330.407500


In [7]:
df_encoded = pd.get_dummies(df, columns=['PaymentMethod', 'OrderStatus'], dtype=int)

print(df_encoded.head())

     OrderID        Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000  2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001  2024-08-23     C75739    Phone         2     151.35   
2  ORD200002  2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003  2023-10-15     C33540    Chair         1     273.19   
4  ORD200004  2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress TrackingNumber  ItemsInCart CouponCode  ...  \
0     928 Main St    TRK37947903            7     SAVE10  ...   
1     823 Main St    TRK91186779            3     SAVE10  ...   
2     512 Main St    TRK42903982            8   FREESHIP  ...   
3     275 Main St    TRK62788070            5     SAVE10  ...   
4     668 Main St    TRK29241424            8     SAVE10  ...   

  PaymentMethod_Cash  PaymentMethod_Credit Card  PaymentMethod_Debit Card  \
0                  0                          0                         1   
1                  0                    

In [8]:
df_encoded['Average_Item_Price'] = df_encoded['TotalPrice'] / df_encoded['Quantity']
df_encoded['Cart_Value_Per_Item'] = df_encoded['TotalPrice'] / df_encoded['ItemsInCart']
df_encoded['TotalPrice_Log'] = np.log1p(df_encoded['TotalPrice'])

print(df_encoded[['Average_Item_Price', 'Cart_Value_Per_Item', 'TotalPrice_Log']].head())

   Average_Item_Price  Cart_Value_Per_Item  TotalPrice_Log
0              570.62           407.585714        7.956512
1              151.35           100.900000        5.716040
2              550.68           344.175000        7.920955
3              273.19            54.638000        5.613821
4              626.01           313.005000        7.826060


In [9]:
numeric_df = df_encoded.select_dtypes(include=[np.number])

correlation_matrix = numeric_df.corr()
print("Correlation Matrix preview:")
print(correlation_matrix[['TotalPrice', 'Average_Item_Price', 'Cart_Value_Per_Item', 'TotalPrice_Log']].head())

df_encoded.to_csv("cleaned_dataset_project1.csv", index=False)
print("\nFinal cleaned dataset exported successfully as 'cleaned_dataset_project1.csv'!")

Correlation Matrix preview:
                    TotalPrice  Average_Item_Price  Cart_Value_Per_Item  \
Quantity              0.615362            0.014079             0.216031   
UnitPrice             0.717232            0.999988             0.748812   
ItemsInCart           0.392579            0.000269            -0.243178   
TotalPrice            1.000000            0.716695             0.696036   
PaymentMethod_Cash    0.001351           -0.025979            -0.028386   

                    TotalPrice_Log  
Quantity                  0.559884  
UnitPrice                 0.759852  
ItemsInCart               0.348793  
TotalPrice                0.874838  
PaymentMethod_Cash       -0.012780  

Final cleaned dataset exported successfully as 'cleaned_dataset_project1.csv'!
